In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "ethiopia"
vehicle = "salt"
scenario = "intervention_25_nrv"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
scenario = "intervention"


In [4]:
def aggregate_by_cause_and_scenario(df):
    result = df.groupby(["scenario", "entity", "input_draw", "wealth_quintile"]).value.sum().groupby(["scenario", "entity", "wealth_quintile"]).mean()
    return result[result.index.get_level_values("entity") != "all_causes"]

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylls = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylls.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,lowest,baseline,10,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,lowest,baseline,10,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,second,baseline,10,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,second,baseline,10,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,middle,baseline,10,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
179995,ylls,cause,other_causes,other_causes,95_plus,severe,middle,baseline,88,0,0.0
179996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,fourth,baseline,88,0,0.0
179997,ylls,cause,other_causes,other_causes,95_plus,severe,fourth,baseline,88,0,0.0
179998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,highest,baseline,88,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        100
intervention    100
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"])
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  fourth             322725.328086
                                  highest            276679.716849
                                  lowest             476817.179521
                                  middle             387054.049816
                                  second             475830.435295
intervention  maternal_disorders  fourth             319227.864634
                                  highest            275496.917052
                                  lowest             471448.096780
                                  middle             383552.700485
                                  second             471402.110537
Name: value, dtype: float64

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylds = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,lowest,baseline,10,0,3.39108
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,lowest,baseline,10,0,0.00000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,lowest,baseline,10,0,0.00000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,lowest,baseline,10,0,0.00000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,lowest,baseline,10,0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...
629995,ylds,cause,pregnancy,parturition,95_plus,severe,highest,baseline,88,0,0.00000
629996,ylds,cause,pregnancy,postpartum,95_plus,severe,highest,baseline,88,0,0.00000
629997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,highest,baseline,88,0,0.00000
629998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,highest,baseline,88,0,0.00000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (pregnancy_ylds[pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])].value == 0).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(lambda df: df[~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])])
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              fourth             20916.357919
                                  highest            15631.086957
                                  lowest             36881.723894
                                  middle             29985.048397
                                  second             42925.952901
              maternal_disorders  fourth                40.194825
                                  highest               40.159149
                                  lowest                61.961172
                                  middle                50.454143
                                  second                59.604511
intervention  anemia              fourth             20063.197807
                                  highest            15053.104542
                                  lowest             34663.236542
                                  middle             28546.322891
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(pregnancy_ylds_by_scenario, fill_value=0)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              fourth              20916.357919
                                  highest             15631.086957
                                  lowest              36881.723894
                                  middle              29985.048397
                                  second              42925.952901
              maternal_disorders  fourth             322765.522912
                                  highest            276719.875999
                                  lowest             476879.140693
                                  middle             387104.503959
                                  second             475890.039806
intervention  anemia              fourth              20063.197807
                                  highest             15053.104542
                                  lowest              34663.236542
                                  middle              28546.322891
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
    )
else:
    neonatal_ylls = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet").assign(value=0).assign(maternal_scenario=lambda x: x.maternal_scenario.replace('intervention', scenario))
    )

neonatal_ylls = neonatal_ylls.rename(columns={"maternal_scenario": "scenario"})
neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,lowest,baseline,intervention,40,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,second,baseline,intervention,40,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,middle,baseline,intervention,40,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,fourth,baseline,intervention,40,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,highest,baseline,intervention,40,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
15995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,lowest,baseline,intervention,49,0,23062.135845
15996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,second,baseline,intervention,49,0,25787.449938
15997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,middle,baseline,intervention,49,0,19360.705372
15998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,fourth,baseline,intervention,49,0,21175.465087


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"] == 0).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"]
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario.reset_index().assign(entity="lbwsg").set_index(neonatal_ylls_by_scenario.index.names).value
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   fourth             1.229759e+07
                      highest            1.078258e+07
                      lowest             1.821689e+07
                      middle             1.547065e+07
                      second             1.839008e+07
intervention  lbwsg   fourth             1.225031e+07
                      highest            1.073867e+07
                      lowest             1.799463e+07
                      middle             1.538232e+07
                      second             1.822922e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(path)
    )
else:
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

non_pregnancy_anemia_ylds

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,424.729902,baseline
1,0.0,0.019178,Female,highest,291.089988,baseline
2,0.0,0.019178,Female,lowest,736.111714,baseline
3,0.0,0.019178,Female,middle,521.550200,baseline
4,0.0,0.019178,Female,second,650.757670,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,75.850598,intervention
496,95.0,125.000000,Male,highest,74.056934,intervention
497,95.0,125.000000,Male,lowest,87.028308,intervention
498,95.0,125.000000,Male,middle,72.260735,intervention


In [17]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0"))
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  fourth             635249.935582
                      highest            449549.169032
                      lowest             955058.158686
                      middle             691583.659226
                      second             804660.940071
intervention  anemia  fourth             621137.215518
                      highest            441284.728136
                      lowest             918953.921992
                      middle             674013.872255
                      second             778716.360060
Name: value, dtype: float64

In [18]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(path)
    )
else:
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(["scenario", "entity", "wealth_quintile"]).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      ntd     lowest             428092.440821
                      second             437751.165498
                      middle             396285.261284
                      fourth             351266.722707
                      highest            308524.090519
intervention  ntd     fourth             205128.469409
                      highest            187792.101519
                      lowest             178968.060340
                      middle             208656.555221
                      second             193302.148375
Name: value, dtype: float64

In [19]:
dalys_by_scenario = pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0).add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0).add(neural_tube_defect_ylls_by_scenario, fill_value=0)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              fourth             6.561663e+05
                                  highest            4.651803e+05
                                  lowest             9.919399e+05
                                  middle             7.215687e+05
                                  second             8.475869e+05
              lbwsg               fourth             1.229759e+07
                                  highest            1.078258e+07
                                  lowest             1.821689e+07
                                  middle             1.547065e+07
                                  second             1.839008e+07
              maternal_disorders  fourth             3.227655e+05
                                  highest            2.767199e+05
                                  lowest             4.768791e+05
                                  middle             3.871045e+05
                          

In [20]:
import pathlib

In [21]:
path = f'./results/{location}/{vehicle}/{scenario}/dalys_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)